<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Learning Loop: Agenten systematisch verbessern
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Mara Vogt nutzt den PMO-Assistenten für quellengebundene Briefings. Wir prüfen mehrere echte Antworten auf Fragen zum bestehenden PMO-Korpus, untersuchen wiederkehrende Fehler und vergleichen eine gezielte Änderung der Anweisung. Anschließend prüfen wir zurückgehaltene Fälle und entscheiden über die Übernahme.

Das Notebook verwendet denselben PMO-Korpus und dieselbe Retrieval-Bibliothek wie M12 und M40. Es baut einen bewusst kleinen PMO-Graphen selbstständig auf; es importiert nicht den vollständigen Referenzgraphen aus M40. Die Modellaufrufe und die Suche sind echt. Der Learning Loop ändert Anweisungen, nicht die Modellgewichte.

In [ ]:
#@title 🛠️ Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q "markitdown[all]"

import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m27_learning_loop"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mermaid, copy_from_github
from genai_lib.model_config import WORKER

setup_api_keys(["OPENAI_API_KEY"], create_globals=False)
check_environment()
get_ipinfo()


# 1 | Übersicht
---

Ein einzelner guter Lauf zeigt noch nicht, ob ein Agent zuverlässig arbeitet. Ein Learning Loop sammelt Ergebnisse mehrerer Fälle, untersucht die Fehler und prüft eine gezielte Änderung gegen die bisherige Version.

**Lernziele:** Du kannst den PMO-Assistenten auf Entwicklungsfällen evaluieren, eine begründete Änderung seiner Anweisung erproben und sie auf zurückgehaltenen Fällen prüfen. Du unterscheidest automatische Hinweise von einer fachlichen Freigabe.

**Voraussetzungen:** State und Nodes aus M08, PMO-Retrieval aus M12 sowie Evaluation aus M13 oder M21. M18 überarbeitet ein einzelnes Ergebnis innerhalb eines Laufs; M27 verbessert die Anweisung für spätere Läufe.

In [ ]:
#@markdown <p><font size="4" color='green'>🔄 Learning Loop im PMO-Assistenten</font></p>
mermaid("""%%{init: {'theme': 'base', 'themeVariables': {'primaryColor': '#e0f2fe', 'primaryTextColor': '#0c4a6e', 'primaryBorderColor': '#0284c7', 'lineColor': '#475569', 'fontFamily': 'Arial'}}}%%
flowchart TD
    A([PMO-Fälle]) --> B[Baseline ausführen]
    B --> C[Ergebnisse prüfen]
    C --> D[Fehler analysieren]
    D --> E[Anweisung ändern]
    E --> F[Variante vergleichen]
    F --> G{Entwicklungsfälle bestanden}
    G -->|Ja| H[Zurückgehaltene Fälle prüfen]
    G -->|Nein| I([Änderung verwerfen])
    H --> J{Keine Regression}
    J -->|Ja| K([Änderung übernehmen])
    J -->|Nein| I
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef stop fill:#fee2e2,stroke:#dc2626,stroke-width:2px,color:#7f1d1d;
    class A input;
    class G,J metric;
    class K output;
    class I stop;
""")


# 2 | PMO-Korpus und Retrieval vorbereiten
---

Wir verwenden den gleichen Korpuspfad und die persistente Chroma-Collection wie die PMO-Referenzimplementierung. Der Korpus liegt in Google Drive. Falls er bereits indexiert wurde, verwendet die Bibliothek den vorhandenen Index; andernfalls indexiert sie die Dokumente.

Die Retrieval-Treffer bilden den gemeinsamen Kontext für Baseline und Variante. So kann ein Unterschied in den Antworten der geänderten Anweisung zugeordnet werden, ohne dass sich zusätzlich die gefundenen Dokumente zwischen den Varianten ändern. Die Retrieval-Qualität selbst prüfen wir separat.

In [ ]:
from pathlib import Path
from google.colab import drive
from genai_lib.pmo_rag import get_pmo_vectorstore, index_pmo_corpus, format_docs

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/Agenten")
corpus_path = drive_root / "02_daten/01_text/korpus_pmo_assistent"
db_path = drive_root / "02_daten/05_sonstiges/chroma_pmo"
corpus_path.mkdir(parents=True, exist_ok=True)
db_path.mkdir(parents=True, exist_ok=True)

copy_from_github(
    source="ralf-42/Agenten/02_daten/01_text/korpus_pmo_assistent",
    target=str(corpus_path),
    mask="*",
)
vectorstore = get_pmo_vectorstore(persist_directory=str(db_path))
indexed_chunks = index_pmo_corpus(vectorstore, str(corpus_path), chunk_size=300, chunk_overlap=30)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Neu indexierte Chunks: {indexed_chunks}")


# 3 | Entwicklungsfälle und zurückgehaltene Fälle
---

Die Fälle orientieren sich an den PMO-Fragen und Quelldateien aus M21. Die ersten Fälle dienen der Analyse und Auswahl einer Änderung. Die zurückgehaltenen Fälle werden erst geprüft, nachdem die Änderung festgelegt ist. In der Praxis werden sie getrennt verwaltet, damit sie die Formulierung der Änderung nicht beeinflussen.

Die erwarteten Quellen sind Anhaltspunkte für den Retrieval-Check. Ein Treffer kann trotzdem fachlich ungeeignet sein; eine Antwort kann auch aus einer anderen gültigen Quelle korrekt belegt sein. Deshalb werden Treffer und Antworten zusätzlich von einem Menschen beurteilt.

In [ ]:
dev_cases = [
    {"id": "dev_1", "query": "Welche Entscheidung gilt für den Prototyp?", "expected_source": "entscheidungsprotokoll_gesamt.pdf", "out_of_corpus": False},
    {"id": "dev_2", "query": "Welche Entscheidung gilt für den produktiven Betrieb?", "expected_source": "entscheidungsprotokoll_gesamt.pdf", "out_of_corpus": False},
    {"id": "dev_3", "query": "Welchen Status hat Risiko R2?", "expected_source": "protokoll_steuerkreis_2026-04-28.pdf", "out_of_corpus": False},
    {"id": "dev_4", "query": "Wie hoch ist das Reisebudget für das nächste Firmenfest?", "expected_source": "", "out_of_corpus": True},
]
holdout_cases = [
    {"id": "test_1", "query": "Wann wurde D2 im Steuerkreis bestätigt?", "expected_source": "protokoll_steuerkreis_2026-04-14.pdf", "out_of_corpus": False},
    {"id": "test_2", "query": "Wer gewinnt die nächste Fußballweltmeisterschaft?", "expected_source": "", "out_of_corpus": True},
]
assert not ({c["id"] for c in dev_cases} & {c["id"] for c in holdout_cases})
print(f"Entwicklungsfälle: {len(dev_cases)}; zurückgehalten: {len(holdout_cases)}")


# 4 | PMO-Agent und Bewertung
---

Der LangGraph erhält die Frage, den bereits abgerufenen Kontext und eine Anweisung. Sein Antwort-Node ruft das konfigurierte Modell auf. Anschließend erfasst ein Prüf-Node einfache Hinweise: Wurde die erwartete Quelle gefunden und genannt? Wird bei einer Frage außerhalb des Korpus die Grenze benannt?

Die Suche wird wie in den Kursstandards ausdrücklich vor dem Graph-Aufruf ausgeführt, dann werden die Dokumente formatiert und an den Agenten übergeben. Semantische Ähnlichkeit liefert auch zu sachfremden Fragen manchmal Treffer. Ob diese Treffer die Antwort tatsächlich stützen, entscheidet nicht der Retrieval-Score, sondern muss anhand der Ausschnitte und Antworten überprüft werden.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

llm = init_chat_model(WORKER)

class LearningState(TypedDict, total=False):
    case: dict
    instruction: str
    context: str
    sources: list[str]
    answer: str
    evaluation: dict



In [ ]:
def answer_node(state: LearningState) -> dict:
    if not state["context"].strip():
        return {"answer": "Nicht im Korpus."}
    result = llm.invoke([
        SystemMessage(content=state["instruction"]),
        HumanMessage(content=f"Frage: {state['case']['query']}\n\nKorpusauszüge:\n{state['context']}"),
    ], config={"run_name": "m27_pmo_antwort"})
    return {"answer": result.text}

def evaluation_node(state: LearningState) -> dict:
    case, answer = state["case"], state["answer"]
    sources = state["sources"]
    if case["out_of_corpus"]:
        check = "nicht im korpus" in answer.casefold()
        retrieval = None
    else:
        expected = case["expected_source"].casefold()
        retrieval = any(expected in source.casefold() for source in sources)
        check = expected in answer.casefold()
    return {"evaluation": {"id": case["id"], "retrieval_hit": retrieval,
                           "answer_check": check, "answer": answer, "sources": sources}}



In [ ]:
agent_builder = StateGraph(LearningState)
agent_builder.add_node("answer", answer_node)
agent_builder.add_node("evaluation", evaluation_node)
agent_builder.add_edge(START, "answer")
agent_builder.add_edge("answer", "evaluation")
agent_builder.add_edge("evaluation", END)
agent = agent_builder.compile()


In [ ]:
def prepare_cases(cases: list[dict]) -> list[dict]:
    prepared = []
    for case in cases:
        docs = retriever.invoke(case["query"])
        context = format_docs(docs)
        sources = [doc.metadata.get("source", "").replace("\\", "/").rsplit("/", 1)[-1] for doc in docs]
        prepared.append({"case": case, "context": context, "sources": sources})
    return prepared

def run_cases(prepared: list[dict], instruction: str) -> list[dict]:
    return [agent.invoke({**item, "instruction": instruction})["evaluation"] for item in prepared]

def show_results(results: list[dict]) -> None:
    for result in results:
        print(f"\n{result['id']} | Retrieval-Treffer: {result['retrieval_hit']} | Antwort-Check: {result['answer_check']}")
        print("Gefundene Quellen:", ", ".join(result["sources"]) or "keine")
        print("Antwort:", result["answer"])
    print(f"\nAntwort-Checks bestanden: {sum(r['answer_check'] for r in results)}/{len(results)}")


Ein Quellenname in der Antwort beweist keine Quellenbindung. Ein `False` kann an einem fehlenden Treffer, einem fehlenden Quellenhinweis oder einer anderen berechtigten Quelle liegen. Für Out-of-Corpus-Fälle prüft der einfache Test nur die Formulierung „Nicht im Korpus“; zusätzliche unbelegte Aussagen müssen manuell erkannt werden.

# 5 | Baseline ausführen und Fehler untersuchen
---

Die Baseline enthält bereits eine Quellenpflicht und eine Korpusgrenze. Sie wird nicht absichtlich verschlechtert. Ein tatsächlicher Unterschied zwischen den Varianten ist deshalb möglich, aber nicht garantiert.

In [ ]:
baseline_instruction = (
    "Du bist der PMO-Assistent. Beantworte Fragen nur anhand der Korpusauszüge. "
    "Nenne deine Quellen. Wenn die Auszüge die Frage nicht beantworten, schreibe: Nicht im Korpus."
)
prepared_dev = prepare_cases(dev_cases)
baseline_dev = run_cases(prepared_dev, baseline_instruction)
show_results(baseline_dev)


Untersuche fehlgeschlagene Checks und lies die Antworten. Vergleiche sie mit den abgerufenen Ausschnitten in `prepared_dev`. Trenne dabei Retrieval-Fehler von Antwortfehlern: Wenn die benötigte Quelle nicht gefunden wurde, kann eine bessere Antwortanweisung die fehlende Evidenz nicht ersetzen.

In [ ]:
for result, item in zip(baseline_dev, prepared_dev):
    if not result["answer_check"] or result["retrieval_hit"] is False:
        print(f"\n--- {result['id']} ---")
        print("Frage:", item["case"]["query"])
        print("Kontext:\n", item["context"])
        print("Antwort:\n", result["answer"])


# 6 | Eine Änderung wählen und vergleichen
---

Eine mögliche Änderung fordert explizit, jede sachliche Aussage mit einem Dateinamen zu verbinden und bei unzureichender Evidenz keine Vermutung anzuhängen. Sie ist ein begründeter Vorschlag, keine garantierte Verbesserung. Passe sie anhand der Entwicklungsanalyse an; verändere möglichst nur einen Aspekt und halte die übrigen Einstellungen konstant.

In [ ]:
candidate_instruction = (
    baseline_instruction +
    " Ordne jede sachliche Aussage einem Dateinamen aus den Korpusauszügen zu. "
    "Reichen die Auszüge nicht aus, gib keine Vermutung aus."
)
candidate_dev = run_cases(prepared_dev, candidate_instruction)
show_results(candidate_dev)

def compare_results(baseline: list[dict], candidate: list[dict]) -> dict:
    before = {r["id"]: r for r in baseline}
    after = {r["id"]: r for r in candidate}
    if len(before) != len(baseline) or len(after) != len(candidate) or not before or before.keys() != after.keys():
        raise ValueError("Beide Läufe brauchen dieselben eindeutigen Fall-IDs.")
    return {
        "improvements": [i for i in before if not before[i]["answer_check"] and after[i]["answer_check"]],
        "regressions": [i for i in before if before[i]["answer_check"] and not after[i]["answer_check"]],
    }

dev_comparison = compare_results(baseline_dev, candidate_dev)
print("Entwicklungsvergleich:", dev_comparison)


Ein automatischer Treffervergleich ist nur ein erster Hinweis. Beurteile insbesondere, ob die Aussagen von den angeführten Auszügen gedeckt sind und ob Antworten unnötig verweigert werden. Vergleiche wiederholte Läufe, wenn die Modellausgaben schwanken. Wähle die Variante erst nach dieser Prüfung; ändere sie nicht anhand der zurückgehaltenen Fälle.

# 7 | Zurückgehaltene Fälle und Entscheidung
---

Die nächste Zelle führt beide festgelegten Varianten auf denselben zurückgehaltenen Fällen aus. Beurteile anschließend Antworten, Quellen und Regressionen. Die Fallzahl ist für eine belastbare Freigabe zu klein; in der Praxis sind mehr unabhängige Fälle und fachliche Reviews nötig.

In [ ]:
prepared_holdout = prepare_cases(holdout_cases)
baseline_holdout = run_cases(prepared_holdout, baseline_instruction)
candidate_holdout = run_cases(prepared_holdout, candidate_instruction)
print("Baseline:")
show_results(baseline_holdout)
print("Variante:")
show_results(candidate_holdout)
print("Testvergleich:", compare_results(baseline_holdout, candidate_holdout))


**Entscheidung:** Übernimm die Änderung nur, wenn sie nach fachlicher Prüfung hilft und keine wichtigen Fälle verschlechtert. Das Notebook ändert keine produktiven Anweisungen automatisch. Nach der Auswertung sind diese Fälle nicht mehr unabhängig zurückgehalten und müssen für spätere Änderungen durch neue ergänzt werden.

# A | Aufgaben
---

**Grundlagen:** Wähle einen Entwicklungsfall. Vergleiche Frage, Retrieval-Treffer, Antwort und automatischen Check. Erkläre, was der Check zeigt und was er nicht zeigt.

**Erledigt, wenn:** Du einen Fall anhand des tatsächlichen Ausschnitts fachlich beurteilt und die Grenze des Checks benannt hast.

**Aufbau:** Untersuche einen Fall ohne erwartete Korpusantwort. Enthält die Ausgabe neben der Korpusgrenze unbelegte Zusatzbehauptungen? Ergänze einen einfachen Test für einen konkreten beobachteten Fehler und erkläre seine Grenzen.

**Erledigt, wenn:** Dein zusätzlicher Test mit einer passenden und einer fehlerhaften Beispielantwort geprüft wurde.

**Vertiefung:** Entwirf eine alternative Anweisungsänderung anhand der Entwicklungsfälle und vergleiche sie mit der Baseline. Prüfe eine neue, zuvor nicht verwendete Gruppe zurückgehaltener Fälle. Dokumentiere Verbesserungen, Regressionen und die Grenzen deiner Schlussfolgerung.

**Praxis-Transfer:** Mara Vogt muss über die Übernahme entscheiden. Stelle die wichtigsten Antworten mit Quellen, problematische Fälle und deine Empfehlung zusammen. Begründe, ob weitere fachliche Prüfung erforderlich ist.